# Predictive Anayltics: Support Vector Machines with Regression for Community Areas

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [61]:
%load_ext cuml.accel
from run_config import PATHS

The cuml.accel extension is already loaded. To reload it, use:
  %reload_ext cuml.accel


In [62]:
import os
os.environ["LD_LIBRARY_PATH"] = "/mnt/c/Users/bkran/Documents/AAA/Group-3-AAA/.venv/lib64/python3.12/site-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

import cuml
print(cuml.__version__)

26.06.00


In [63]:
TRAIN_SAMPLE = 70_000 
GRID_SAMPLE = 70_000 # if validation set over GRID_SEARCH use only GRID_SEARCH rows of data due to runtime issues, for grid search
SPATIAL_UNIT = "COMMUNITY_AREAS" # COMMUNITY_AREAS
SPATIAL_ENCODING = "latlong" # options: latlong, onehot
TIME_UNIT = "1H" # options: 1H, 4H, 24H

In [64]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv" # same file in full/sample
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [65]:
import pandas as pd
import geopandas as gpd
import numpy as np
import polars as pl
from shapely import wkt

# cuml
from cuml import SVR
from cuml import LinearSVR

#sklearn
#from sklearn.svm import SVR 
#from sklearn.svm import LinearSVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.experimental import enable_halving_search_cv # noqa
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.utils import resample
from sklearn.base import clone

# joblib
from joblib import load, dump
from joblib import Memory




## Preparations

In [66]:
INPUT = PATHS.train_test_dir

In [67]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [68]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [69]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-12-22 10:00:00,12,1,10,-0.500000,8.660254e-01,0.000000,1.000000,0.500000,-8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
1,2025-08-18 04:00:00,8,1,4,-0.500000,-8.660254e-01,0.000000,1.000000,0.866025,5.000000e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
2,2025-09-19 23:00:00,9,5,23,-0.866025,-5.000000e-01,-0.433884,-0.900969,-0.258819,9.659258e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
3,2025-11-22 07:00:00,11,6,7,-0.866025,5.000000e-01,-0.974928,-0.222521,0.965926,-2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
4,2025-09-24 20:00:00,9,3,20,-0.866025,-5.000000e-01,0.974928,-0.222521,-0.866025,5.000000e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626467,2025-08-20 05:00:00,8,3,5,-0.500000,-8.660254e-01,0.974928,-0.222521,0.965926,2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
626468,2025-04-11 06:00:00,4,5,6,1.000000,6.123234e-17,-0.433884,-0.900969,1.000000,6.123234e-17,...,0.0,0.0,0.0,0.0,0.0,308.37,20.558,5.13,55.31,Mobile
626469,2025-06-06 21:00:00,6,5,21,0.500000,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips
626470,2025-11-10 00:00:00,11,1,0,-0.866025,5.000000e-01,0.000000,1.000000,0.000000,1.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.00,0.00,No trips


In [70]:
if len(train_df) > TRAIN_SAMPLE:
   train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=40)

In [71]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
192563,2025-06-30 06:00:00,6,1,6,0.5,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
201427,2025-08-22 21:00:00,8,5,21,-0.5,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
118253,2026-04-08 15:00:00,4,3,15,1.0,6.123234e-17,0.974928,-0.222521,-0.707107,-7.071068e-01,...,6.0,1108.13,8.148015,0.0,54.0,8712.28,64.060882,16.05,172.5,Credit Card
363574,2025-12-06 23:00:00,12,6,23,-0.5,8.660254e-01,-0.974928,-0.222521,-0.258819,9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
198821,2026-02-23 11:00:00,2,1,11,0.5,8.660254e-01,0.000000,1.000000,0.258819,-9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips


In [72]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [73]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [74]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
192563,2025-06-30 06:00:00,6,1,6,0.500000,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
201427,2025-08-22 21:00:00,8,5,21,-0.500000,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
118253,2026-04-08 15:00:00,4,3,15,1.000000,6.123234e-17,0.974928,-0.222521,-0.707107,-7.071068e-01,...,6.0,1108.13,8.148015,0.0,54.0,8712.28,64.060882,16.05,172.5,Credit Card
363574,2025-12-06 23:00:00,12,6,23,-0.500000,8.660254e-01,-0.974928,-0.222521,-0.258819,9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
198821,2026-02-23 11:00:00,2,1,11,0.500000,8.660254e-01,0.000000,1.000000,0.258819,-9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269279,2026-02-18 09:00:00,2,3,9,0.500000,8.660254e-01,0.974928,-0.222521,0.707107,-7.071068e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
299486,2025-10-30 11:00:00,10,4,11,-1.000000,-1.836970e-16,0.433884,-0.900969,0.258819,-9.659258e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
561642,2026-01-18 16:00:00,1,7,16,0.000000,1.000000e+00,-0.781831,0.623490,-0.866025,-5.000000e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips
579169,2026-01-27 20:00:00,1,2,20,0.000000,1.000000e+00,0.781831,0.623490,-0.866025,5.000000e-01,...,0.0,0.00,0.000000,0.0,0.0,0.00,0.000000,0.00,0.0,No trips


In [75]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: community_area")

    census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
    census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

    census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  


    gdf_proj = gdf.to_crs(epsg=3435)
    gdf["lon"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).x
    gdf["lat"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).y

    tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

    for df in (train_df, val_df, test_df):
        df["community_area"] = df["community_area"].astype(str).str.zfill(2)
        df["lat"] = df["community_area"].map(tract_centroids["lat"])
        df["lon"] = df["community_area"].map(tract_centroids["lon"])

        # sanity check, catch silent join failures early
        n_missing_lat = df["lat"].isna().sum()
        n_missing_lon = df["lon"].isna().sum()
        if n_missing_lat or n_missing_lon:
            print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")


Encoding: latlong and Unit: community_area


In [76]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else:
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Onehot

In [77]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)

Create y

In [78]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [79]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
192563,0.500000,-8.660254e-01,0.000000,1.000000,1.000000,6.123234e-17,0,1.637913,26.0,3.0,...,0,0,0,0,1,0,0,0.029036,-0.745148,0.666267
201427,-0.500000,-8.660254e-01,-0.433884,-0.900969,-0.707107,7.071068e-01,0,4.769125,19.0,2.0,...,1,0,0,0,1,0,0,0.029966,-0.744927,0.666473
118253,1.000000,6.123234e-17,0.974928,-0.222521,-0.707107,-7.071068e-01,0,2.797738,116.0,10.0,...,0,0,0,0,0,1,0,0.027324,-0.742926,0.668815
363574,-0.500000,8.660254e-01,-0.974928,-0.222521,-0.258819,9.659258e-01,0,1.637913,26.0,3.0,...,0,0,1,0,1,0,0,0.029036,-0.745148,0.666267
198821,0.500000,8.660254e-01,0.000000,1.000000,0.258819,-9.659258e-01,0,1.637913,26.0,3.0,...,0,1,0,0,1,0,0,0.029036,-0.745148,0.666267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269279,0.500000,8.660254e-01,0.974928,-0.222521,0.707107,-7.071068e-01,0,12.999586,35.0,7.0,...,0,0,0,0,1,0,0,0.029599,-0.743722,0.667833
299486,-1.000000,-1.836970e-16,0.433884,-0.900969,0.258819,-9.659258e-01,0,11.602461,86.0,11.0,...,0,0,0,0,1,0,0,0.030755,-0.744351,0.667080
561642,0.000000,1.000000e+00,-0.781831,0.623490,-0.866025,-5.000000e-01,0,6.613966,34.0,18.0,...,0,0,1,0,1,0,0,0.029717,-0.744428,0.667041
579169,0.000000,1.000000e+00,0.781831,0.623490,-0.866025,5.000000e-01,0,12.919814,21.0,6.0,...,0,0,0,0,1,0,0,0.030372,-0.746140,0.665097


### Grid Search

In [80]:
model = SVR()

In [81]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVR(max_iter=100_000,tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()), # only included for 24H due to small dataset
    ('svm', SVR(max_iter=50_000, tol=1e-2))
], memory=memory)

# regressor
ttr_linear = TransformedTargetRegressor(regressor=pipe_linear, transformer=StandardScaler())
ttr_kernel = TransformedTargetRegressor(regressor=pipe_kernel, transformer=StandardScaler())

# parameter for each kernel # excluded C=100, C=10, 0,001 excluded via testing due to convergance issues
param_grid_linear = {
    "regressor__svm__C": [1, 10, 30, 100], # 4h: , 1h: 24h: 
    "regressor__svm__epsilon": [0.01, 0.05, 0.1, 0.3],
}

param_grid_rbf_sigmoid = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["rbf", "sigmoid"],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300],  # for 24h 10, 30 and 4h, 1h: 100, 300
}

param_grid_poly = {
    "regressor__svm__C": [1, 10],
    "regressor__svm__epsilon": [0.01, 0.05, 0.1],
    "regressor__feature_map__kernel": ["poly"],
    "regressor__feature_map__degree": [3, 4],
    "regressor__feature_map__gamma": [0.01, 0.1, 1],
    "regressor__feature_map__n_components": [100, 300], # for 24h 10, 30 and 4h, 1h: 100, 300
}

grids = {}
configs = [
    ("linear", ttr_linear, param_grid_linear),
    ("rbf_sigmoid", ttr_kernel, param_grid_rbf_sigmoid),
    ("poly", ttr_kernel, param_grid_poly),
]

# doing gridsearch on all
for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2, # changed to 2 due to runtime issues
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.27757926626240653 best params: {'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.3}
rbf_sigmoid best score: 0.8169864224919874 best params: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}
poly best score: 0.8097531877916924 best params: {'regressor__feature_map__degree': 3, 'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'poly', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}
Overall best: rbf_sigmoid {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}


In [82]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'regressor__feature_map__gamma': 0.01, 'regressor__feature_map__kernel': 'rbf', 'regressor__feature_map__n_components': 300, 'regressor__svm__C': 10, 'regressor__svm__epsilon': 0.1}
Best CV score: 0.8169864224919874


### Train Model

In [83]:
best_model = grid_search.best_estimator_

In [84]:
# Train SVR 

best_model.fit(X_train, y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","Pipeline(memo...svm', SVR())])"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: str or callable, default='rbf'Kernel map to be approximated. A callable should accept two argumentsand the keyword arguments passed to this object as `kernel_params`, andshould return a floating point number.",'rbf'
,"gamma gamma: float, default=NoneGamma parameter for the RBF, laplacian, polynomial, exponential chi2and sigmoid kernels. Interpretation of the default value is left tothe kernel; see the documentation for sklearn.metrics.pairwise.Ignored by other kernels.",0.01
,"coef0 coef0: float, default=NoneZero coefficient for polynomial and sigmoid kernels.Ignored by other kernels.",None


In [85]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [86]:
y_pred

array([ 1.41853228,  2.17870013,  1.02099648, ..., 58.99766752,
        2.70464758, 10.38063139], shape=(134904,))

In [87]:
# Evaluation metrics
result = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2 Score": r2_score(y_test, y_pred),
}

In [88]:
df = pd.DataFrame(y_pred)
df.to_csv("../models/svm/model_" + SPATIAL_UNIT + "_" + "_" + TIME_UNIT + ".csv")
pd.DataFrame([result]).to_csv("../models/svm/result_" + SPATIAL_UNIT + "_" + TIME_UNIT + ".csv", index=False)

In [89]:
dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")

['../models/svm/model_COMMUNITY_AREAS_1H_svr.joblib']

In [90]:
# save model

dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/svm/grid_COMMUNITY_AREAS_1H_svr.joblib']